In [1]:
import numpy, torch
print("NumPy:", numpy.__version__)
print("Torch:", torch.__version__)
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
import holidays
print("Alles importiert ✔")

NumPy: 1.26.4
Torch: 2.2.2+cu121


C:\Users\kaiws\anaconda3\envs\tft\lib\site-packages\lightning_fabric\__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


Alles importiert ✔


In [2]:
import warnings; warnings.filterwarnings("ignore")
import os
import torch
import pandas as pd
import numpy as np
import gc
import holidays
from pathlib import Path
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE as NF_MAE
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.INFO)
from pytorch_lightning.callbacks import RichProgressBar
# --- 1. CONFIG & SETUP ---
torch.set_float32_matmul_precision('high')
STATION = "Aotizhongxin"
FREQ = "h"
H_TFT = 72
INPUT_SIZE = 168
# Pfad-Setup
DATA_DIR = Path("../data/prepared")
os.makedirs(DATA_DIR / "basis", exist_ok=True)

# --- 2. HILFSFUNKTIONEN (Daten & Zeit) ---
def lade(variante, station=STATION):
    """Läd Daten lokal. Falls nicht vorhanden, stelle sicher, dass sie im Pfad liegen."""
    path = DATA_DIR / variante / f"prophet_train_{station}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Daten nicht gefunden unter {path}. Bitte Datenaufbereitung durchführen.")
    tr = pd.read_csv(path, parse_dates=["ds"])
    te = pd.read_csv(DATA_DIR / variante / f"prophet_test_{station}.csv", parse_dates=["ds"])
    return tr, te

def add_feiertag(df):
    """Fügt Feiertage hinzu (CN-Kalender)."""
    cn_holidays = holidays.country_holidays("CN", years=range(2013, 2018))
    d = df.copy()
    d["feiertag"] = d["ds"].dt.date.astype("O").map(lambda t: 1 if t in cn_holidays else 0)
    return d

def add_time_features(df):
    """Zyklische Zeit-Features für Saisonalität."""
    d = df.copy()
    d['hour_sin'] = np.sin(2 * np.pi * d['ds'].dt.hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * d['ds'].dt.hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * d['ds'].dt.dayofyear / 365.25)
    d['day_cos'] = np.cos(2 * np.pi * d['ds'].dt.dayofyear / 365.25)
    return d

def zu_nf_format(df, regressoren, feiertag_flag, unique_id=STATION):
    d = add_time_features(df)
    if feiertag_flag:
        d = add_feiertag(d)
    d.insert(0, "unique_id", unique_id)
    time_features = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos']
    cols = ["unique_id", "ds", "y"] + regressoren + time_features + (["feiertag"] if feiertag_flag else [])
    return d[cols]

def regularize(df, spalten=None):
    """Lückenloses Stundenraster für NeuralForecast."""
    spalten = spalten or [c for c in df.columns if c != "ds"]
    d = df.set_index("ds").sort_index()
    voll = pd.date_range(d.index.min(), d.index.max(), freq=FREQ)
    d = d.reindex(voll)
    for c in spalten:
        if c == "feiertag": d[c] = d[c].ffill().bfill()
        else: d[c] = d[c].interpolate(limit_direction="both")
    return d.reset_index()

# --- 3. TFT TRAININGSLOGIK ---
def tft_fit_predict(train, test, regressoren=None, feiertag=False, log=False):
    regressoren = regressoren or []
    exog = regressoren + ['hour_sin', 'hour_cos', 'day_sin', 'day_cos'] + (["feiertag"] if feiertag else [])
    
    gesamt = regularize(pd.concat([train, test], ignore_index=True), spalten=["y"] + regressoren)
    voll_nf = zu_nf_format(gesamt, regressoren, feiertag)
    if log: voll_nf = voll_nf.assign(y=np.log1p(voll_nf["y"]))

    model = TFT(h=H_TFT, input_size=INPUT_SIZE, hist_exog_list=exog, futr_exog_list=exog,
                loss=NF_MAE(), max_steps = 50, batch_size=4)
    
    nf = NeuralForecast(models=[model], freq=FREQ)
    trainer_kwargs = {"accelerator": "gpu", "devices": 1, "precision": "16-mixed","enable_progress_bar": True,"refresh_rate": 1,"callbacks": [RichProgressBar()]}
    
    cv_df = nf.cross_validation(df=voll_nf, n_windows=2, step_size=H_TFT, trainer_kwargs=trainer_kwargs)
    
    rueck = (lambda a: np.expm1(a)) if log else (lambda a: a)
    cv_df["yhat"] = rueck(cv_df["TFT"]).clip(lower=0)
    out = test[["ds", "y"]].merge(cv_df[["ds", "yhat"]], on="ds")
    
    del model, nf, cv_df
    gc.collect(); torch.cuda.empty_cache()
    return out

# --- 4. EXECUTION ---
# Beispiel: Lade Daten (müssen im Ordner ../data/prepared liegen)
# tr, te = lade("basis")
# result = tft_fit_predict(tr, te, regressoren=[], feiertag=True, log=True)
# print(result.head())

In [3]:
import warnings; warnings.filterwarnings("ignore")
import os
import torch
import pandas as pd
import numpy as np
import gc
import holidays
from pathlib import Path
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE as NF_MAE

# --- 1. CONFIGURATION ---
torch.set_float32_matmul_precision('high')
STATION = "Aotizhongxin"
FREQ = "h"
H_TFT = 72
INPUT_SIZE = 168
DATA_DIR = Path("../data/prepared")
REGRESSOREN = ["TEMP", "DEWP", "PRES", "WSPM", "RAIN", "wd_sin", "wd_cos"]

# --- 2. PREPROCESSING FUNKTIONEN ---
def add_feiertag(df):
    cn_holidays = holidays.country_holidays("CN", years=range(2013, 2018))
    d = df.copy()
    d["feiertag"] = d["ds"].dt.date.astype("O").map(lambda t: 1 if t in cn_holidays else 0)
    return d

def add_time_features(df):
    d = df.copy()
    d['hour_sin'] = np.sin(2 * np.pi * d['ds'].dt.hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * d['ds'].dt.hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * d['ds'].dt.dayofyear / 365.25)
    d['day_cos'] = np.cos(2 * np.pi * d['ds'].dt.dayofyear / 365.25)
    return d

def zu_nf_format(df, regressoren, feiertag_flag, unique_id=STATION):
    d = add_time_features(df)
    if feiertag_flag: d = add_feiertag(d)
    d.insert(0, "unique_id", unique_id)
    cols = ["unique_id", "ds", "y"] + regressoren + ['hour_sin', 'hour_cos', 'day_sin', 'day_cos'] + (["feiertag"] if feiertag_flag else [])
    return d[cols]

def regularize(df, spalten=None):
    """Bringt einen Frame auf ein lücken-freies Stundenraster."""
    spalten = spalten or [c for c in df.columns if c != "ds"]
    
    # Sicherstellen, dass ds Datetime ist, bevor wir es zum Index machen
    df['ds'] = pd.to_datetime(df['ds'])
    
    d = df.set_index("ds").sort_index()
    voll = pd.date_range(d.index.min(), d.index.max(), freq=FREQ)
    d = d.reindex(voll)
    
    for c in spalten:
        if c == "feiertag":
            d[c] = d[c].ffill().bfill()
        else:
            d[c] = d[c].interpolate(limit_direction="both")
    
    d.index.name = "ds"
    # Hier der entscheidende Fix: reset_index() stellt 'ds' als Spalte wieder her
    return d.reset_index()

# --- 3. TRAINING & INFERENZ ---
def tft_fit_predict(train, test, regressoren=None, feiertag=False, log=False):
    regressoren = regressoren or []
    exog = regressoren + ['hour_sin', 'hour_cos', 'day_sin', 'day_cos'] + (["feiertag"] if feiertag else [])
    
    gesamt = regularize(pd.concat([train, test], ignore_index=True), spalten=["y"] + regressoren)
    voll_nf = zu_nf_format(gesamt, regressoren, feiertag)
    if log: voll_nf = voll_nf.assign(y=np.log1p(voll_nf["y"]))

    model = TFT(h=H_TFT, input_size=INPUT_SIZE, hist_exog_list=exog, futr_exog_list=exog,
                loss=NF_MAE(),max_steps=500, batch_size=8)
    print(f"Trainingsbeispiele pro Fenster: {len(voll_nf) - INPUT_SIZE}")
    nf = NeuralForecast(models=[model], freq=FREQ)
    trainer_kwargs = {"accelerator": "gpu", "devices": 1, "precision": "16-mixed","enable_progress_bar": True,"refresh_rate": 1,"callbacks": [RichProgressBar()]}
    
    # Cross Validation
    cv_df = nf.cross_validation(df=voll_nf, n_windows=10, step_size=H_TFT, trainer_kwargs=trainer_kwargs)
    
    rueck = (lambda a: np.expm1(a)) if log else (lambda a: a)
    cv_df["yhat"] = rueck(cv_df["TFT"]).clip(lower=0)
    out = test[["ds", "y"]].merge(cv_df[["ds", "yhat"]], on="ds")
    
    # Cleanup
    del model, nf, cv_df
    gc.collect(); torch.cuda.empty_cache()
    return out

# --- 4. EXECUTION TEIL ---
# Konfigurationsleiter
konfigs_tft = {
    "Basis": dict(variante="basis", regressoren=[], feiertag=False, log=False),
    "Behandelt": dict(variante="behandelt", regressoren=REGRESSOREN, feiertag=True, log=True)
}

ergebnisse = {}
for name, cfg in konfigs_tft.items():
    print(f"Starte Training: {name}...")
    tr = pd.read_csv(DATA_DIR / cfg["variante"] / f"prophet_train_{STATION}.csv", parse_dates=["ds"])
    te = pd.read_csv(DATA_DIR / cfg["variante"] / f"prophet_test_{STATION}.csv", parse_dates=["ds"])
    
    out = tft_fit_predict(tr, te, regressoren=cfg["regressoren"], feiertag=cfg["feiertag"], log=cfg["log"])
    ergebnisse[name] = out
    print(f"Fertig: {name} | MAE: {np.mean(np.abs(out['y'] - out['yhat'])):.2f}")

# Finaler Report
for name, out in ergebnisse.items():
    print(f"{name} - RMSE: {np.sqrt(np.mean((out['y'] - out['yhat'])**2)):.2f}")

Seed set to 1


Starte Training: Basis...
Trainingsbeispiele pro Fenster: 34896


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: C:\Users\kaiws\Claude\Projects\Business Prognosis\capstone projekt\notebooks\lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                    | Type                     | Params
---------------------------------------------------------------------
0 | loss                    | MAE                      | 0     
1 | padder_train            | ConstantPad1d            | 0     
2 | scaler                  | TemporalNorm             | 0     
3 | embedding               | TFTEmbedding             | 2.3 K 
4 | temporal_encoder        | TemporalCovariateEncoder | 1.5 M 
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 256 K 
6 | output_adapter          | Linear                   | 129   
---------------------------------------------------------------------
1.7 M     Trainable params
0         N

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=500` reached.


TypeError: TimeSeriesDataModule.__init__() got an unexpected keyword argument 'trainer_kwargs'